# CyborgDB Service + Python SDK + Postgres — End‑to‑End Quickstart

This notebook installs the **CyborgDB Service** (`cyborgdb-service`), the **CyborgDB Python SDK** (`cyborgdb`), and **sentence-transformers**; launches the service as a subprocess; creates embeddings for a small example dataset; **upserts** them; **queries**; shows the **top match**; and finally **shuts down** the service.

**Prerequisites:** You need **PostgreSQL or Redis** available and a **CyborgDB API key**.

Read our docs [here](https://docs.cyborg.co) and get a free API key [here](https://cyborgdb.co).

## 1) Install dependencies

In [9]:
%pip install --quiet --upgrade cyborgdb-service cyborgdb sentence-transformers ipywidgets

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


## 2) Configure environment variables

In [10]:
import os, getpass

# Get environment variables
API_KEY = os.environ.get("CYBORGDB_API_KEY") or getpass.getpass("Enter CYBORGDB_API_KEY: ")
DB_TYPE = "postgres"
CONN = os.environ.get("POSTGRES_CONNECTION_STRING") or input("Enter CYBORGDB_CONNECTION_STRING for Postgres: ").strip()

print(CONN)

# Set downstream environment variables
os.environ["CYBORGDB_DB_TYPE"] = DB_TYPE
os.environ["CYBORGDB_CONNECTION_STRING"] = CONN

print("Environment variables set.")

host=localhost port=5432 dbname=postgres user=postgres password=qwe329
Environment variables set.


## 3) Launch the service and verify health
Starts `cyborgdb-service` as a subprocess and polls `/v1/health`.

In [11]:
import subprocess, time, requests, os, signal

SERVICE_CMD = ["cyborgdb-service"]
service_proc = subprocess.Popen(SERVICE_CMD, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=os.environ.copy())
CYBORGDB_SERVICE_PID = service_proc.pid
print(f"Launched cyborgdb-service pid={CYBORGDB_SERVICE_PID}")

base_url = "http://localhost:8000"
for _ in range(60):
    try:
        resp = requests.get(f"{base_url}/v1/health", timeout=2)
        if resp.status_code == 200:
            print("Service healthy.")
            break
    except Exception as e:
        pass
    time.sleep(1)
else:
    print("Service did not become healthy. Check DB connectivity and env settings.")


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Launched cyborgdb-service pid=26872
Service healthy.


## 4) Create a client and an index
Using IVF‑Flat with dimension 384 and embedding model `all-MiniLM-L6-v2`.

In [13]:
from cyborgdb import Client, IndexIVFFlat

# Create a CyborgDB Client via the SDK
client = Client("http://localhost:8000", api_key=API_KEY)

# Create an Encrypted Index
index_key = client.generate_key()
config = IndexIVFFlat(dimension=384, n_lists=100, metric="cosine")
index_name = "quickstart_" + __import__("uuid").uuid4().hex[:8]
index = client.create_index(index_name, index_key, config, embedding_model="all-MiniLM-L6-v2")

print("\nIndex ready:", index_name)

SSL verification is disabled. Not recommended for production.



Index ready: quickstart_4f3be168


## 5) Create embeddings for example items

In [ ]:
from sentence_transformers import SentenceTransformer
st = SentenceTransformer("all-MiniLM-L6-v2")

texts = [
    "Customer record: Jane Doe, SSN 123-45-6789, account balance $42,500.",
    "Confidential contract clause: Early termination incurs a 15% penalty.",
    "Employee medical note: Allergic to penicillin, prescribed alternative treatment.",
    "Internal API key for payment processor: sk_live_9f2ab03...",
    "Patient diagnosis report: Stage II hypertension, recommended lifestyle changes.",
    "M&A draft: Proposed acquisition of Acme Corp valued at $120M.",
    "Proprietary algorithm details: Vector compression technique reduces storage by 70%.",
    "Confidential salary spreadsheet entry: VP of Engineering — $210,000 base + equity.",
    "Internal security incident: Unauthorized access attempt on 2024-08-14 from IP 10.23.4.55.",
    "Board meeting notes: Strategic expansion into EU markets starting Q3 2025.",
    "Private customer support ticket: User cannot reset MFA, mobile number compromised.",
    "Intellectual property filing draft: New blockchain consensus mechanism design.",
    "Financial forecast: Projected burn rate $1.2M/month, runway until Jan 2027.",
    "Confidential R&D experiment results: New material has 30% higher tensile strength.",
    "Internal credentials: VPN password expired, rotated on 2025-09-01.",
    "Executive email: Discussing layoffs in non-core divisions by year-end.",
    "Private legal memo: Pending litigation risk assessment for patent infringement.",
    "Internal roadmap: Launch of secure AI analytics product slated for April 2026.",
    "Encrypted backup manifest: Contains full copy of production customer database.",
    "Confidential HR record: Employee disciplinary warning issued 2025-07-10.",
]

vecs = st.encode(texts, normalize_embeddings=True).tolist()

embedded_docs = []
for i, (c, v) in enumerate(zip(texts, vecs), 1):
    embedded_docs.append({"id": f"doc_{i}", "vector": v, "metadata": {"contents": c}})

print("Embedded", len(embedded_docs), "docs.")


## 6) Upsert into CyborgDB

In [ ]:
index.upsert(embedded_docs)
print("Upserted", len(embedded_docs), "items.")

## 7) Query and show the top match

In [ ]:
query_text = "When will we run out of cash according to our budget?"
res = index.query(query_contents=query_text, top_k=3)

top_results = res[0] if res else []

for r in top_results:
    print(f"id={r.get('id')} distance={r.get('distance'):.4f} contents={r.get('contents','')[:80]}...")
if top_results:
    print("\nTop match:", top_results[0].get("id"))
    # Get the full document from the database
    print("Document:", top_results[0].get("metadata", {}).get("contents", ""))
    

### (Optional) Query by vector

In [ ]:
qv = st.encode(["Make embeddings smaller"], normalize_embeddings=True)
vres = index.query(query_vectors=qv[0], top_k=3)
if vres and vres[0]:
    print("Vector query top id:", vres[0][0]["id"])
    print("Document:", vres[0][0].get("metadata", {}).get("contents", ""))

## 8) Shut down the service

In [ ]:
import os, signal, time
try:
    pid = CYBORGDB_SERVICE_PID
except NameError:
    pid = None
if pid:
    print(f"Terminating cyborgdb-service pid={pid} ...")
    try:
        os.kill(pid, signal.SIGTERM)
    except Exception as e:
        print("SIGTERM failed:", e)
    print("Shutdown attempted.")
else:
    print("No service PID found.")